# Assignment 04: Vector Spatial Analysis

## BIO597 Spatial Analysis of Biodiversity

This assignment asks you to apply spatial joins, buffers, nearest-feature analysis, and polygon overlay to Maine amphibian occurrences.

## Deliverable

Submit a completed notebook containing code, maps, and short interpretations for at least three spatial questions.

## Learning goals

By the end of this assignment, you should be able to:

* connect occurrence points to polygon attributes with a spatial join
* summarize records and species using a new spatial unit
* create and interpret a distance buffer
* calculate distance to the nearest line feature
* use intersection or difference to compare polygon layers

## Data

Use `Amphibians_ME/Amphibians_ME.csv` for occurrence records. The conserved-land, HUC12 watershed, and major-road shapefiles are in `../labs/ME_ShapeFiles`.

Read the `README.md` in that directory before beginning. In particular, remember that the road layer contains primary and secondary roads rather than every local road.

## 1. Import packages

Import `pandas` as `pd` and `geopandas` as `gpd`.

In [ ]:
# Import the two packages here.

## 2. Load the occurrence table

Read the tab-delimited amphibian file. Display the first five rows after it loads.

In [ ]:
amphibians = pd.read_csv(
    "Amphibians_ME/Amphibians_ME.csv",
    sep=__________,
    low_memory=False,
)

__________

## 3. Keep the columns needed for spatial analysis

Keep the record identifier, species, longitude, latitude, coordinate uncertainty, and year. Then display the shape of the smaller table.

In [ ]:
columns_to_keep = [
    "gbifID",
    "species",
    "decimalLongitude",
    "decimalLatitude",
    "coordinateUncertaintyInMeters",
    "year",
]

# Subset amphibians using columns_to_keep.
# Display the number of rows and columns.

## 4. Prepare the coordinate columns

Convert longitude and latitude to numeric values with `pd.to_numeric()`. Use `errors="coerce"` so unusable values become missing values.

In [ ]:
# Convert decimalLongitude to numeric.
# Convert decimalLatitude to numeric.

## 5. Remove unusable records

Remove rows missing a species name, longitude, or latitude. Use `.copy()` to create an independent table.

In [ ]:
amphibians = amphibians.dropna(
    subset=[__________, __________, __________]
).copy()

amphibians.shape

## 6. Create an occurrence GeoDataFrame

Use `gpd.points_from_xy()` and assign `EPSG:4326`.

In [ ]:
amphibians_gdf = gpd.GeoDataFrame(
    amphibians,
    geometry=gpd.points_from_xy(
        amphibians[__________],
        amphibians[__________],
    ),
    crs=__________,
)

amphibians_gdf.head()

## 7. Load the three supporting layers

Use `gpd.read_file()` to load:

* `Maine_Conserved_Lands.shp`
* `Maine_HUC12_Watersheds.shp`
* `Maine_Primary_Secondary_Roads.shp`

In [ ]:
shape_directory = "../labs/ME_ShapeFiles"

# Load the conserved-land polygons as conserved.
# Load the watershed polygons as watersheds.
# Load the road lines as roads.

## 8. Inspect the layers

For each GeoDataFrame, display or print its CRS and geometry types. Write one sentence explaining why these details matter.

In [ ]:
# Inspect CRS and geometry types here.

**Why this matters:**

## 9. Project every layer

Project all four GeoDataFrames to `EPSG:26919`. Use clear variable names ending in `_projected`.

In [ ]:
maine_crs = "EPSG:26919"

# Project the occurrences, conserved lands, watersheds, and roads.

# Spatial question 1: Which records occur inside conserved lands?

This section is partly guided. Use a point-in-polygon spatial join with the predicate `within`.

In [ ]:
conserved_columns = ["PARCEL_NAM", "CONS1_TYPE", "PUB_ACCESS", "geometry"]

occurrences_in_conserved = gpd.sjoin(
    __________,
    __________[conserved_columns],
    how="inner",
    predicate=__________,
)

occurrences_in_conserved.head()

## 10. Count unique records and species

Conservation polygons can overlap. First remove repeated `gbifID` values created by overlapping polygon matches. Then calculate:

* the number of unique occurrence records inside conserved lands
* the number of unique species inside conserved lands

In [ ]:
# Remove duplicate gbifID values.
# Calculate the two requested counts.

## 11. Map and interpret conserved-land occurrences

Create an interactive map of the matching points colored by species. Include the conserved parcel name in the tooltip.

In [ ]:
# Convert the matching points to EPSG:4326 and map them with .explore().

**Interpretation:** In two or three sentences, describe the result and identify one limitation of treating an occurrence inside a conserved-land boundary as evidence that conservation caused the occurrence.

# Spatial question 2: How are occurrences distributed among watersheds?

Choose **three amphibian species** for this analysis. Write their scientific names below and create a GeoDataFrame containing only those species.

**Species 1:**

**Species 2:**

**Species 3:**

In [ ]:
selected_species = [
    "____________________________",
    "____________________________",
    "____________________________",
]

# Filter amphibians_projected using .isin(selected_species).

## 12. Assign selected occurrences to HUC12 watersheds

Use `gpd.sjoin()` with the watershed columns `huc12`, `name`, and `geometry`.

In [ ]:
# Spatially join your selected occurrence points to watersheds.
# Display the first five rows of the result.

## 13. Summarize by watershed

Create a table with one row per HUC12 watershed. Include:

* the number of unique occurrence records
* the number of unique selected species

Use the Lab 04 workflow as a model, but write this code yourself.

In [ ]:
# Group by huc12 and name.
# Calculate unique records and unique species.
# Reset the index.

## 14. Map the watershed result

Merge your summary table back into the watershed polygons. Make a map colored by either record count or species count.

In [ ]:
# Merge the summary with watersheds_projected.
# Convert to EPSG:4326 and map with .explore().

**Interpretation:** Which watersheds have the most records or species? Give one ecological explanation and one sampling explanation for the pattern.

# Spatial question 3: Which records are within 500 m of a major road?

Create a 500 m buffer around the projected roads. Combine the overlapping buffers with `union_all()`, then place that geometry in a one-row GeoDataFrame.

In [ ]:
road_buffer_geometry = roads_projected.buffer(__________)
road_buffer_union = road_buffer_geometry.__________()

road_buffer = gpd.GeoDataFrame(
    {"buffer_m": [__________]},
    geometry=[road_buffer_union],
    crs=maine_crs,
)

## 15. Find occurrences inside the road buffer

Use a spatial join to find all occurrence points within the combined road buffer.

In [ ]:
# Spatially join amphibians_projected to road_buffer.
# Display the shape of the result.

## 16. Summarize the road-buffer result

Report:

* the number of unique species with at least one record within 500 m of a major road
* the number of records within 500 m for each species
* the percentage of all occurrence records that are within 500 m

In [ ]:
# Calculate the three requested summaries.

**Interpretation:** Does this result demonstrate that amphibians prefer roads? Explain why or why not, and mention how observer access could affect the pattern.

# Spatial question 4: How far are records from the nearest major road?

Use `gpd.sjoin_nearest()` to add the nearest road and its distance to every occurrence. Keep `FULLNAME`, `MTFCC`, and `geometry` from the road layer. Name the distance column `distance_to_road_m`.

In [ ]:
road_columns = ["FULLNAME", "MTFCC", "geometry"]

# Complete the nearest spatial join.

## 17. Compare nearest-road distance among your three species

Filter the nearest-road result to your three selected species. Calculate the median distance to the nearest major road for each species.

In [ ]:
# Filter with .isin(selected_species).
# Group by species and calculate the median distance_to_road_m.

**Interpretation:** Which selected species has records closest to major roads? Give at least one reason this could reflect the observation process instead of species biology.

# Spatial question 5: Conserved area in one watershed

Choose one HUC12 watershed from your summary. Store its HUC12 code below, then create a one-row or small GeoDataFrame containing that watershed.

In [ ]:
chosen_huc12 = "____________"

# Filter watersheds_projected to chosen_huc12.

## 18. Intersect conserved lands with the chosen watershed

Use `gpd.overlay()` with `how="intersection"`. Calculate the total intersecting conserved area in square kilometers.

In [ ]:
# Intersect conserved_projected with your chosen watershed.
# Sum polygon area and convert square meters to square kilometers.

## 19. Find the part outside conserved lands

Dissolve the intersecting conservation polygons, then use `gpd.overlay()` with `how="difference"` to retain the part of the watershed outside those polygons.

In [ ]:
# Dissolve the intersecting conserved polygons.
# Calculate the difference between the watershed and dissolved polygons.

## 20. Map and interpret the overlay

Make an interactive map showing either the conserved intersection or the area outside conserved lands. Report the approximate percentage of the watershed inside conserved lands.

In [ ]:
# Calculate the percentage.
# Convert your chosen result to EPSG:4326 and map it.

**Interpretation:** What does intersection retain? What does difference retain? State one limitation of the conserved-land boundaries.

## Final reflection

Answer each question in two or three sentences.

1. Which spatial operation changed the unit of analysis most clearly?
2. Which of your results is most sensitive to the choice of CRS?
3. Which result is most likely to be affected by uneven sampling effort?
4. Propose one additional ecological or conservation question that could be answered with these layers.

## Submit your work

Run the notebook from top to bottom. Make sure all maps display and all written interpretations are complete. Then commit and push the completed notebook to your class GitHub repository.

```bash
git add docs/assignments/Assignment-04-VectorSpatialAnalysis.ipynb
git commit -m "Complete vector spatial analysis assignment"
git push
```